# `model.py` — Multi-Aspect Sentiment Model (RoBERTa + Dependency GCN)

## Purpose
Defines the **core neural architecture** of the ClearView sentiment model. This is the main research contribution — a RoBERTa-based encoder enhanced with:
1. **Aspect-aware Multi-Head Attention** — uses a learnable aspect embedding as the query vector, so the model attends to different parts of the text for different aspects (colour vs. smell vs. price).
2. **Dependency GCN** — a Graph Convolutional Network built from the syntactic dependency parse tree of each review, enabling the model to understand which words are syntactically connected to opinion targets.
3. **Aspect-specific classifiers** — separate classification heads per aspect.

## Architecture Diagram
```
Text Input
   │
   ▼
[RoBERTa Encoder]  ← contextual token embeddings (768-dim)
   │
   ├─────────────────────────────────┐
   ▼                                 ▼
[Aspect-Aware MHA]              [Dep GCN] ← dependency parse edges
   │  (aspect query vector)          │
   ▼                                 │
[Aspect Representation]        [GCN Pooled Output]
   │                                 │
   └──────────── concat ─────────────┘
                     │
              [Final Classifier]
                     │
              [Logits: neg/neu/pos]
```

## Classes
| Class | Description |
|-------|-------------|
| `AspectAwareRoBERTa` | RoBERTa + aspect-guided MHA + per-aspect classifiers |
| `AspectOrientedDepGCN` | 2-layer GCN with aspect-gating over dependency edges |
| `MultiAspectSentimentModel` | Combines both into the full model |
| `create_model()` | Factory function — use this to instantiate the model |

In [1]:
import time
_start_time = time.time()
from tqdm.auto import tqdm
print("⏳ Starting: Loading dependencies and libraries...")
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import RobertaModel   # Pre-trained RoBERTa backbone

print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Loading dependencies and libraries.")


⏳ Starting: Loading dependencies and libraries...
🕒 Done in 6.57s
✅ Completed: Loading dependencies and libraries.


## 1. Aspect-Aware RoBERTa

The core idea is to use the **aspect identity as the attention query**. Instead of using the `[CLS]` token (which summarises the whole text), we compute an attention-weighted representation over all tokens using the aspect's learnable embedding as the query.

This forces the model to aggregate token information *specific to the aspect being predicted* rather than producing one generic text vector.

In [ ]:
print("⏳ Starting: Defining class AspectAwareRoBERTa...")
from tqdm.auto import tqdm
import time
_start_time = time.time()
class AspectAwareRoBERTa(nn.Module):
    """
    RoBERTa with aspect-specific attention. Improvement over generic [CLS] pooling.
    """
    def __init__(self, roberta_model='roberta-base', num_aspects=7, num_classes=3,
                 hidden_dim=768, dropout=0.1, use_aspect_attention=True, use_shared_classifier=False):
        super(AspectAwareRoBERTa, self).__init__()

        self.use_aspect_attention  = use_aspect_attention   # Ablation 2 flag
        self.use_shared_classifier = use_shared_classifier  # Ablation 5 flag

        self.roberta = RobertaModel.from_pretrained(roberta_model)  # Load pre-trained weights

        # Learnable query vectors: one per aspect (e.g. smell, colour, price...)
        # Xavier uniform init keeps initial gradient magnitudes stable during training.
        self.aspect_embeddings = nn.Embedding(num_aspects, hidden_dim)
        nn.init.xavier_uniform_(self.aspect_embeddings.weight)

        if use_aspect_attention:
            # Multi-head attention: aspect embedding is the query, token embeddings are keys/values
            self.aspect_attention = nn.MultiheadAttention(
                embed_dim=hidden_dim, num_heads=8, dropout=dropout, batch_first=True
            )

        self.layer_norm = nn.LayerNorm(hidden_dim)  # Stabilise after attention
        self.dropout    = nn.Dropout(dropout)

        # Ablation 5: shared head OR per-aspect heads
        if use_shared_classifier:
            # Single MLP for all aspects (simpler, but less aspect-specific)
            self.shared_classifier = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(),
                nn.Dropout(dropout), nn.Linear(hidden_dim // 2, num_classes)
            )
        else:
            # One MLP head per aspect — allows each aspect to learn its own decision boundary
            self.aspect_classifiers = nn.ModuleList([
                nn.Sequential(
                    nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(),
                    nn.Dropout(dropout), nn.Linear(hidden_dim // 2, num_classes)
                ) for _ in range(num_aspects)
            ])

        self.num_aspects = num_aspects
        self.hidden_dim  = hidden_dim

    def forward(self, input_ids, attention_mask, aspect_id, return_token_embeddings=False):
        # 1. Run through pre-trained RoBERTa
        roberta_output = self.roberta(
            input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True
        )
        hidden_states = roberta_output.last_hidden_state  # (batch, seq_len, 768)

        if self.use_aspect_attention:
            # 2a. Aspect-guided attention 
            aspect_query = self.aspect_embeddings(aspect_id).unsqueeze(1)  # (batch, 1, 768)
            attended_output, attention_weights = self.aspect_attention(
                query=aspect_query, key=hidden_states, value=hidden_states,
                key_padding_mask=~attention_mask.bool()  # Mask out padding tokens
            )
            aspect_representation = attended_output.squeeze(1)  # (batch, 768)
        else:
            # 2b. Ablation 2: plain CLS pooling — no aspect awareness
            aspect_representation = hidden_states[:, 0, :]  # Take the [CLS] token
            attention_weights = torch.ones(
                hidden_states.size(0), 1, hidden_states.size(1), device=hidden_states.device
            ) / hidden_states.size(1)  # Uniform fake weights for interface compatibility

        aspect_representation = self.layer_norm(aspect_representation)
        aspect_representation = self.dropout(aspect_representation)

        # 3. Classify using the aspect-specific (or shared) head
        predictions = []
        for i in tqdm(range(input_ids.size(0)), desc="I progress"):
            if self.use_shared_classifier:
                pred = self.shared_classifier(aspect_representation[i])
            else:
                pred = self.aspect_classifiers[aspect_id[i].item()](aspect_representation[i])
            predictions.append(pred)

        predictions = torch.stack(predictions)  # (batch, num_classes)

        if return_token_embeddings:
            return predictions, attention_weights.squeeze(1), aspect_representation, hidden_states
        return predictions, attention_weights.squeeze(1), aspect_representation

print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Defining class AspectAwareRoBERTa.")


⏳ Starting: Defining class AspectAwareRoBERTa...
🕒 Done in 0.00s
✅ Completed: Defining class AspectAwareRoBERTa.


## 2. Aspect-Oriented Dependency GCN

The GCN takes the **dependency parse edges** of the review (e.g. `likes→colour`, `colour→vibrant`) and propagates information along these edges. Each GCN layer:
1. Aggregates messages from neighbours (scatter-sum over edges).
2. Applies an **aspect gate** that decides how much of the GCN update to keep vs. keep the original token representation — this ensures that for a `colour` query, colour-related tokens receive more GCN-propagated information.

In [3]:
print("⏳ Starting: Defining class AspectOrientedDepGCN...")
from tqdm.auto import tqdm
import time
_start_time = time.time()
class AspectOrientedDepGCN(nn.Module):
    """
    2-layer GCN over dependency parse graph with aspect-gated residual connections.
    """
    def __init__(self, hidden_dim=768, num_layers=2, dropout=0.1):
        super(AspectOrientedDepGCN, self).__init__()

        # One linear transform per GCN layer (propagates aggregated messages)
        self.gcn_layers  = nn.ModuleList([nn.Linear(hidden_dim, hidden_dim) for _ in range(num_layers)])

        # Aspect gate: given the GCN output and the aspect embedding, output a gate in [0,1]
        # gate=1 → use GCN update completely; gate=0 → keep original token embedding
        self.aspect_gate = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),  # Concat [gcn, aspect_query] → hidden
            nn.Sigmoid()                             # Element-wise gate
        )
        self.layer_norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(num_layers)])
        self.dropout     = nn.Dropout(dropout)
        self.num_layers  = num_layers

    def forward(self, token_embeddings, edge_index, aspect_embedding):
        """
        Args:
            token_embeddings: (seq_len, 768) — RoBERTa token vectors
            edge_index:       (2, num_edges) — [src, dst] dependency edges
            aspect_embedding: (768,) — the query aspect's embedding (acts as the gate context)
        """
        x         = token_embeddings
        num_nodes = x.size(0)

        for i in tqdm(range(self.num_layers), desc="I progress"):
            if edge_index.size(1) > 0:
                src, dst = edge_index[0], edge_index[1]
                messages  = x[src]                        # Gather source node features
                aggregated = torch.zeros_like(x)
                # scatter_add_: for each edge, add src message to dst node's bucket
                aggregated.scatter_add_(0, dst.unsqueeze(1).expand_as(messages), messages)
                x_gcn = F.relu(self.gcn_layers[i](aggregated))
            else:
                x_gcn = F.relu(self.gcn_layers[i](x))    # No edges: just transform in place

            # Aspect gate: modulate the update proportionally to aspect relevance
            aspect_expanded = aspect_embedding.unsqueeze(0).expand(num_nodes, -1)
            gate   = self.aspect_gate(torch.cat([x_gcn, aspect_expanded], dim=-1))  # (seq_len, 768)

            # Gated residual: gate selects between GCN output and original embedding
            x = gate * x_gcn + (1 - gate) * x
            x = self.layer_norms[i](x)
            x = self.dropout(x)

        return x   # (seq_len, 768) enhanced token representations

print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Defining class AspectOrientedDepGCN.")


⏳ Starting: Defining class AspectOrientedDepGCN...
🕒 Done in 0.00s
✅ Completed: Defining class AspectOrientedDepGCN.


## 3. Full MultiAspectSentimentModel
Wires together the two modules above. The GCN path is optional (controlled by `use_dependency_gcn` in config).

In [ ]:
print("⏳ Starting: Defining class MultiAspectSentimentModel...")
from tqdm.auto import tqdm
import time
_start_time = time.time()
class MultiAspectSentimentModel(nn.Module):
    """Full model: RoBERTa + Aspect Attention + Dependency GCN."""

    def __init__(self, config):
        super(MultiAspectSentimentModel, self).__init__()
        self.config  = config
        model_config = config['model']
        self.use_gcn = model_config.get('use_dependency_gcn', True)  # Ablation 1 flag

        self.aspect_aware_roberta = AspectAwareRoBERTa(
            roberta_model=model_config['roberta_model'],
            num_aspects=model_config['num_aspects'],
            num_classes=model_config['num_classes'],
            hidden_dim=model_config['hidden_dim'],
            dropout=model_config['dropout'],
            use_aspect_attention=model_config.get('use_aspect_attention', True),
            use_shared_classifier=model_config.get('use_shared_classifier', False),
        )

        if self.use_gcn:
            self.dep_gcn = AspectOrientedDepGCN(
                hidden_dim=model_config['hidden_dim'],
                num_layers=model_config['gcn_layers'],
                dropout=model_config['dropout']
            )
            # Final classifier: takes concatenated attention + GCN representations
            self.final_classifier = nn.Sequential(
                nn.Linear(model_config['hidden_dim'] * 2, model_config['hidden_dim']),
                nn.ReLU(), nn.Dropout(model_config['dropout']),
                nn.Linear(model_config['hidden_dim'], model_config['num_classes'])
            )

    def forward(self, input_ids, attention_mask, aspect_id, edge_index=None,
                token_mask=None, return_attention=False):
        need_token_embeddings = self.use_gcn and edge_index is not None

        # Run through RoBERTa + aspect attention
        if need_token_embeddings:
            attn_preds, attn_weights, aspect_repr, token_embs = self.aspect_aware_roberta(
                input_ids, attention_mask, aspect_id, return_token_embeddings=True
            )
        else:
            attn_preds, attn_weights, aspect_repr = self.aspect_aware_roberta(
                input_ids, attention_mask, aspect_id
            )

        # If no GCN, return the attention-based predictions directly
        if not need_token_embeddings:
            if return_attention: return attn_preds, attn_weights, aspect_repr, None
            return attn_preds

        # Apply Dependency GCN per sample in the batch
        gcn_outputs = []
        for i in tqdm(range(input_ids.size(0)), desc="I progress"):
            if edge_index[i].size(1) > 0:
                aspect_emb = self.aspect_aware_roberta.aspect_embeddings(aspect_id[i])
                gcn_out    = self.dep_gcn(token_embs[i], edge_index[i], aspect_emb)

                # Mean-pool GCN output over non-padding positions
                mask_expanded = attention_mask[i].unsqueeze(-1).float()
                gcn_pooled    = (gcn_out * mask_expanded).sum(0) / (mask_expanded.sum(0) + 1e-9)
                gcn_outputs.append(gcn_pooled)
            else:
                gcn_outputs.append(torch.zeros_like(aspect_repr[i]))  # No edges → zero contribution

        gcn_output = torch.stack(gcn_outputs)            # (batch, 768)
        combined   = torch.cat([aspect_repr, gcn_output], dim=-1)  # (batch, 1536)

        final_predictions = self.final_classifier(combined)  # (batch, num_classes)

        if return_attention: return final_predictions, attn_weights, aspect_repr, gcn_output
        return final_predictions

    def get_num_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def create_model(config):
    """Factory function — always use this instead of instantiating directly."""
    model      = MultiAspectSentimentModel(config)
    num_params = model.get_num_parameters()
    print(f'Created model with {num_params:,} trainable parameters')
    return model

print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Defining class MultiAspectSentimentModel.")


⏳ Starting: Defining class MultiAspectSentimentModel...
🕒 Done in 0.00s
✅ Completed: Defining class MultiAspectSentimentModel.


## Quick Test / Verification

Runs a **CPU-only forward pass** with random inputs — no GPU or checkpoint required.
Verifies:
-  alone produces  logits
-  with GCN produces the same shape
- Attention weights have the right shape 
-  factory prints parameter count and returns a model

> The first run downloads  weights (~500 MB) if not cached.

In [5]:
print("⏳ Starting: Creating model...")
import time
_start_time = time.time()
BATCH_SIZE = 2
SEQ_LEN    = 32   # short sequence for fast test
NUM_ASPECTS = 7
NUM_CLASSES = 3
HIDDEN_DIM  = 768

# ── Minimal config dict ────────────────────────────────────────────────────
config = {'model': {
    'roberta_model'        : 'roberta-base',
    'num_aspects'          : NUM_ASPECTS,
    'num_classes'          : NUM_CLASSES,
    'hidden_dim'           : HIDDEN_DIM,
    'dropout'              : 0.1,
    'gcn_layers'           : 2,
    'use_dependency_gcn'   : True,
    'use_aspect_attention' : True,
    'use_shared_classifier': False,
}}

# ── Create model via factory ────────────────────────────────────────────────
model = create_model(config)
model.eval()  # disable dropout for deterministic output

# ── Synthetic input tensors ─────────────────────────────────────────────────
input_ids      = torch.randint(0, 50265, (BATCH_SIZE, SEQ_LEN))   # random token IDs
attention_mask = torch.ones(BATCH_SIZE, SEQ_LEN, dtype=torch.long)
aspect_id      = torch.tensor([0, 1])  # two different aspects

# ── Test 1: No GCN (edge_index=None) ───────────────────────────────────────
with torch.no_grad():
    preds = model(input_ids, attention_mask, aspect_id)

assert preds.shape == (BATCH_SIZE, NUM_CLASSES), f"Expected ({BATCH_SIZE},{NUM_CLASSES}), got {preds.shape}"
print(f"[OK] No-GCN forward pass: {preds.shape}")

# ── Test 2: With GCN (dummy edge_index) ────────────────────────────────────
# edge_index is a list (one per sample) of (2, num_edges) tensors
edge_index = [
    torch.tensor([[0, 1, 2], [1, 2, 3]], dtype=torch.long),  # sample 0: 3 edges
    torch.tensor([[0, 1],    [1, 2]],    dtype=torch.long),  # sample 1: 2 edges
]

with torch.no_grad():
    preds_gcn, attn, asp_repr, gcn_out = model(
        input_ids, attention_mask, aspect_id,
        edge_index=edge_index, return_attention=True
    )

assert preds_gcn.shape == (BATCH_SIZE, NUM_CLASSES), f"GCN pred shape wrong: {preds_gcn.shape}"
assert attn.shape[0]   == BATCH_SIZE,               f"Attn batch dim wrong: {attn.shape}"
assert gcn_out.shape   == (BATCH_SIZE, HIDDEN_DIM), f"GCN output shape wrong: {gcn_out.shape}"
print(f"[OK] GCN forward pass: preds={preds_gcn.shape}, attn={attn.shape}, gcn={gcn_out.shape}")

print("")
print("All model.py tests PASSED.")

print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Creating model.")


⏳ Starting: Creating model...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Created model with 132,637,464 trainable parameters


I progress:   0%|          | 0/2 [00:00<?, ?it/s]

[OK] No-GCN forward pass: torch.Size([2, 3])


I progress:   0%|          | 0/2 [00:00<?, ?it/s]

I progress:   0%|          | 0/2 [00:00<?, ?it/s]

I progress:   0%|          | 0/2 [00:00<?, ?it/s]

I progress:   0%|          | 0/2 [00:00<?, ?it/s]

[OK] GCN forward pass: preds=torch.Size([2, 3]), attn=torch.Size([2, 32]), gcn=torch.Size([2, 768])

All model.py tests PASSED.
🕒 Done in 1.39s
✅ Completed: Creating model.
